# Data Cleaning

In [145]:
# import libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, trim

## 1. Reading parquet files

In [146]:
# define paths to parquet files

forenames = "../data/bronze/df_forenames/"
countries = "../data/bronze/df_countryCode/"

In [147]:
# set a spark session

def create_spark_session(app_name:str):
    spark = SparkSession.builder \
        .appName(app_name) \
        .master("local[*]") \
        .getOrCreate()
    return spark

In [148]:
# Create Spark session for silver stage

spark = create_spark_session("silver_pipeline")

In [149]:
df_fn = spark.read.parquet(forenames)
df_fn.show(5)

+------------+------+-------+-----+
|    forename|gender|country|count|
+------------+------+-------+-----+
|       Yasif|  NULL|     LY|    2|
| رحومه رحومه|  NULL|     LY|    2|
| رحومة رحومة|  NULL|     LY|    2|
|          Nä|  NULL|     LY|    2|
|العمر الضياع|  NULL|     LY|    2|
+------------+------+-------+-----+
only showing top 5 rows


In [150]:
df_fn.printSchema()

root
 |-- forename: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- country: string (nullable = true)
 |-- count: integer (nullable = true)



In [151]:
df_cc = spark.read.parquet(countries)
df_cc.show(5)

+------------+--------------------+
|country_code|        country_name|
+------------+--------------------+
|          AE|United Arab Emirates|
|          AF|         Afghanistan|
|          AL|             Albania|
|          AO|              Angola|
|          AR|           Argentina|
+------------+--------------------+
only showing top 5 rows


In [152]:
df_cc.printSchema()

root
 |-- country_code: string (nullable = true)
 |-- country_name: string (nullable = true)



## 2. Data Exploration 

In [153]:
df_fn.forename.isNull()

Column<'isNull(forename)'>

In [154]:
# function to show null values per column in a specific dataframe

def count_null_values(df):
    for c in df.columns:
        print(c,": ", df.filter(col(c).isNull()).count())

In [155]:
# Shownull values for forename dataframe
count_null_values(df_fn)

forename :  291
gender :  2419135
country :  27572
count :  0


In [156]:
df_fn.count()

12470920

In [157]:
# Shownull values for country code dataframe
count_null_values(df_cc)

country_code :  0
country_name :  0


In [158]:
# show the number of records stored on country code dataframe
df_cc.count()

105

## 2. Data Cleaning

In [159]:
# drop null values on forenames dataframe
df_fn_cleaned = df_fn.dropna()
count_null_values(df_fn_cleaned)

forename :  0
gender :  0
country :  0
count :  0


In [160]:
# Start to work with country code dataframe to normalice country names
"""
Conver country name to lower case and remove blank spaces at star and at the end of the name
"""
df_cc = df_cc.withColumn('country_name', lower(col('country_name')))
df_cc = df_cc.withColumn('country_name', trim(col('country_name')))
df_cc.show(200)

+------------+--------------------+
|country_code|        country_name|
+------------+--------------------+
|          AE|united arab emirates|
|          AF|         afghanistan|
|          AL|             albania|
|          AO|              angola|
|          AR|           argentina|
|          AT|             austria|
|          AZ|          azerbaijan|
|          BD|          bangladesh|
|          BE|             belgium|
|          BF|        burkina faso|
|          BG|            bulgaria|
|          BH|             bahrain|
|          BI|             burundi|
|          BN|              brunei|
|          BO|             bolivia|
|          BR|              brazil|
|          BW|            botswana|
|          CA|              canada|
|          CH|         switzerland|
|          CL|               chile|
|          CM|            cameroon|
|          CN|               china|
|          CO|            colombia|
|          CR|          costa rica|
|          CY|              

In [161]:
# Define a list with LATAM country names, to seek it on country_naem dataframe
"""
In this spet I ensure that the resulting name match with LATAM names
"""
latam_countries = [
    "argentina", "bolivia", "brazil", "chile", "colombia", "costa rica",
    "cuba", "ecuador", "el salvador", "guatemala", "honduras", "mexico",
    "nicaragua", "panama", "paraguay", "peru", "puerto rico", "dominican republic",
    "uruguay", "venezuela"
]

df_cc_latam = df_cc.filter(col('country_name').rlike("|".join(latam_countries)))
df_cc_latam.show()

+------------+------------+
|country_code|country_name|
+------------+------------+
|          AR|   argentina|
|          BO|     bolivia|
|          BR|      brazil|
|          CL|       chile|
|          CO|    colombia|
|          CR|  costa rica|
|          EC|     ecuador|
|          GT|   guatemala|
|          HN|    honduras|
|          MX|      mexico|
|          PA|      panama|
|          PE|        peru|
|          PR| puerto rico|
|          SV| el salvador|
|          UY|     uruguay|
+------------+------------+



In [162]:
# Count the number of records on latam name dataframe
"""
As a result we got 15 records
"""
df_cc_latam.count()

15

In [163]:
# Retreive country code from df_cc_latam, with these values we can filter df_fn_cleaned
country_code_list = [row['country_code'] for row in df_cc_latam.select('country_code').collect()]
country_code_list

['AR',
 'BO',
 'BR',
 'CL',
 'CO',
 'CR',
 'EC',
 'GT',
 'HN',
 'MX',
 'PA',
 'PE',
 'PR',
 'SV',
 'UY']

In [164]:
# Show new number of records on forenames dataframe
df_fn_cleaned.count()

10025658

In [170]:
df_fn_cleaned = df_fn_cleaned.filter(col('country').isin(country_code_list))
df_fn_cleaned.show()

+---------+------+-------+------+
| forename|gender|country| count|
+---------+------+-------+------+
|     Jose|     M|     MX|139094|
|     Juan|     M|     MX|129572|
|     Luis|     M|     MX|127585|
|   Carlos|     M|     MX|124848|
|    Jesus|     M|     MX|101158|
|    Jorge|     M|     MX| 90002|
|Alejandro|     M|     MX| 84172|
|   Daniel|     M|     MX| 78366|
|    Maria|     F|     MX| 76883|
|   Miguel|     M|     MX| 73827|
|    Angel|     M|     MX| 72212|
|   Manuel|     M|     MX| 69090|
|  Eduardo|     M|     MX| 67992|
| Fernando|     M|     MX| 65460|
|Francisco|     M|     MX| 64676|
|  Antonio|     M|     MX| 64325|
|   Javier|     M|     MX| 63019|
|    David|     M|     MX| 56303|
|      Ana|     F|     MX| 54916|
|  Ricardo|     M|     MX| 52890|
+---------+------+-------+------+
only showing top 20 rows


In [171]:
# Show how many records has every country code
df_fn_cleaned.groupBy('country').count().show()

+-------+------+
|country| count|
+-------+------+
|     MX|230800|
|     EC| 11559|
|     SV|   446|
|     PA| 54875|
|     PE|223930|
|     PR|  6373|
|     GT| 45977|
|     CL|151531|
|     BR|188422|
|     CR| 38041|
|     CO|331917|
|     BO| 75105|
|     AR| 55951|
|     HN|  1240|
|     UY| 31612|
+-------+------+



In [167]:
df_fn_cleaned.count()

10025658